# Generation of master table

Containing LFC information as well as editing information for all screens and base means.

In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
import scipy.stats
import os
from adjustText import adjust_text
import matplotlib.patheffects as PathEffects
import warnings
warnings.filterwarnings('ignore')
plt.rc('font', family='Helvetica')

In [97]:
# loading in editing information

library = pd.read_csv('../../source_data/02_library/CDK_library_final.csv')


#-----load single amino-acid variants (SAVs) breakdown for each subpool------
SAV_ABE1 = pd.read_csv('../../screening_data/04_editing/SAVs/ABE_subpool1.zip')
SAV_CBE1 = pd.read_csv('../../screening_data/04_editing/SAVs/CBE_subpool1.zip')

SAV_ABE2 = pd.read_csv('../../screening_data/04_editing/SAVs/ABE_subpool2.zip')
SAV_CBE2 = pd.read_csv('../../screening_data/04_editing/SAVs/CBE_subpool2.zip')

SAV_ABE3 = pd.read_csv('../../screening_data/04_editing/SAVs/ABE_subpool3.zip')
SAV_CBE3 = pd.read_csv('../../screening_data/04_editing/SAVs/CBE_subpool3.zip')

#-----and load in the HGVSp information------
ABE1 = pd.read_csv('../../screening_data/04_editing/ABE_subpool1_HGVSp.zip')
CBE1 = pd.read_csv('../../screening_data/04_editing/CBE_subpool1_HGVSp.zip')

ABE2 = pd.read_csv('../../screening_data/04_editing/ABE_CDK12_13_HGVSp.zip')
CBE2 = pd.read_csv('../../screening_data/04_editing/CBE_CDK12_13_HGVSp.zip')

ABE3 = pd.read_csv('../../screening_data/04_editing/ABE_CDK2_4_6_HGVSp.zip')
CBE3 = pd.read_csv('../../screening_data/04_editing/CBE_CDK2_4_6_HGVSp.zip')

In [98]:
def mis_non_WT_counter(comb):
    """ 
    input = SAV df for SINGLE gRNA subset
    """
    type_mut = []
    for i, val in comb.iterrows():
        if '*' in val['HGVSp']:
            type_mut.append('Nonsense')
        elif 'WT'==val['HGVSp']:
            type_mut.append('WT')
        else:
            type_mut.append('Missense')

    comb['mutation_class'] = type_mut

    l1 = comb[['mutation_class', '%Reads']].groupby('mutation_class').sum().reset_index()

    if len(l1['mutation_class'])!=3:

        cols = ['WT', 'Nonsense', 'Missense']

        l2 = []
        for j in cols:
            if j not in list(l1['mutation_class']):

                aa = pd.DataFrame(dict(zip(['mutation_class', '%Reads'], [[j], [0]])))
                l2.append(aa)

        l1 = pd.concat(([l1]+l2))

    

    return dict(zip(l1['mutation_class'], l1['%Reads']))

In [99]:
gRNA = np.unique(ABE1['gRNA_id'])[0]

comb = ABE1[ABE1['gRNA_id']=='gRNA_CDK19_targ_6304']

mis_non_WT_counter(comb)

{'Missense': 81.92663443346646, 'WT': 18.07336556653353, 'Nonsense': 0.0}

In [104]:
# compiling editing information
min_perc  = 1

SAVs_ABE = [SAV_ABE1, SAV_ABE2, SAV_ABE3]
SAVs_CBE = [SAV_CBE1, SAV_CBE2, SAV_CBE3]

hgs_ABE = [ABE1, ABE2, ABE3]
hgs_CBE = [CBE1, CBE2, CBE3]

#include WT perc and nonsense perc for each
ABE_holder = []
for i, val in enumerate(SAVs_ABE):

    guides = np.unique(val['gRNA_id'])

    hg_df = hgs_ABE[i]

    subset_holder = []
    for guide in guides:
        subset = val[val['gRNA_id']==guide]

        hg_df_subset = hg_df[hg_df['gRNA_id']==guide]

        mis_dict = mis_non_WT_counter(hg_df_subset)

        subset['WT_perc'] = mis_dict['WT']
        subset['Nonsense_perc'] = mis_dict['Nonsense']
        subset['Missense_perc'] = mis_dict['Missense']

        subset_holder.append(subset)

    #and then remove low efficiency edits
    sav_df = pd.concat(subset_holder)
    sav_df = sav_df[sav_df['%Reads']>=min_perc]

    num_reads_df = hg_df[['gRNA_id', '#Reads']].groupby('gRNA_id').sum().reset_index().rename(columns = {'#Reads':'Total_sensor_reads'})

    a = hg_df[hg_df['HGVSp']!='WT']
    a = a.sort_values(by=['gRNA_id', '%Reads'], ascending=False).drop_duplicates(subset='gRNA_id').reset_index(drop=True).rename(columns = {'HGVSp':'Top_HGVSp', '%Reads':'Top_HGVSp_%Reads'})

    #and finally merge together
    m1 = pd.merge(sav_df, a, on='gRNA_id')
    m2 = pd.merge(m1, num_reads_df, on='gRNA_id')
    m2['Editor']='ABE'
    ABE_holder.append(m2)

#-----and for CBE-----
CBE_holder = []
for i, val in enumerate(SAVs_CBE):

    guides = np.unique(val['gRNA_id'])

    hg_df = hgs_CBE[i]

    subset_holder = []
    for guide in guides:
        subset = val[val['gRNA_id']==guide]

        hg_df_subset = hg_df[hg_df['gRNA_id']==guide]

        mis_dict = mis_non_WT_counter(hg_df_subset)

        subset['WT_perc'] = mis_dict['WT']
        subset['Nonsense_perc'] = mis_dict['Nonsense']
        subset['Missense_perc'] = mis_dict['Missense']

        subset_holder.append(subset)

    #and then remove low efficiency edits
    sav_df = pd.concat(subset_holder)
    sav_df = sav_df[sav_df['%Reads']>=min_perc]
    num_reads_df = hg_df[['gRNA_id', '#Reads']].groupby('gRNA_id').sum().reset_index().rename(columns = {'#Reads':'Total_sensor_reads'})

    a = hg_df[hg_df['HGVSp']!='WT']
    a = a.sort_values(by=['gRNA_id', '%Reads'], ascending=False).drop_duplicates(subset='gRNA_id').reset_index(drop=True).rename(columns = {'HGVSp':'Top_HGVSp', '%Reads':'Top_HGVSp_%Reads'})

    #and finally merge together
    m1 = pd.merge(sav_df, a, on='gRNA_id')
    m2 = pd.merge(m1, num_reads_df, on='gRNA_id')
    m2['Editor']='CBE'
    CBE_holder.append(m2)

In [129]:
#need to extract codons from HGVSp as well
cols = ['gRNA_id','HGVSp', '%Reads', 'Total_sensor_reads', 'Codon', 'Top_HGVSp', 'Top_HGVSp_%Reads','WT_perc', 'Missense_perc', 'Nonsense_perc', 'Editor']

aaa = pd.concat(ABE_holder)
ABE_editing_master = aaa[cols]

ccc = pd.concat(CBE_holder)
CBE_editing_master = ccc[cols]

import re
def extract_numbers(text):
    """Extracts all numbers from the given text."""
    if text=='WT':
        return None
    else:
        return [int(i) for i in re.findall(r'\d+', text)][0]

#add information about mutation type (for SAV of interest) AND codon of top_HGVSp
type_mut = []
type_mut_top = []
codon_top_hgvsp = []

for i, val in ABE_editing_master.iterrows():
    if '*' in val['HGVSp']:
        type_mut.append('Nonsense')
    elif 'WT'==val['HGVSp']:
        type_mut.append('WT')
    else:
        type_mut.append('Missense')


    if '*' in val['Top_HGVSp']:
        type_mut_top.append('Nonsense')
    elif 'WT'==val['Top_HGVSp']:
        type_mut_top.append('WT')
    else:
        type_mut_top.append('Missense')

    c = extract_numbers(val['Top_HGVSp'])
    codon_top_hgvsp.append(c)

ABE_editing_master['Mutation_Class_SAV'] = type_mut
ABE_editing_master['Mutation_Class_Top_HGVSp'] = type_mut_top
ABE_editing_master['Top_HGVSp_Codon'] = codon_top_hgvsp

#add information about mutation type (for SAV of interest) AND codon of top_HGVSp
type_mut = []
type_mut_top = []
codon_top_hgvsp = []
for i, val in CBE_editing_master.iterrows():
    if '*' in val['HGVSp']:
        type_mut.append('Nonsense')
    elif 'WT'==val['HGVSp']:
        type_mut.append('WT')
    else:
        type_mut.append('Missense')

    if '*' in val['Top_HGVSp']:
        type_mut_top.append('Nonsense')
    elif 'WT'==val['Top_HGVSp']:
        type_mut_top.append('WT')
    else:
        type_mut_top.append('Missense')

    c = extract_numbers(val['Top_HGVSp'])
    codon_top_hgvsp.append(c)

CBE_editing_master['Mutation_Class_SAV'] = type_mut
CBE_editing_master['Mutation_Class_Top_HGVSp'] = type_mut_top
CBE_editing_master['Top_HGVSp_Codon'] = codon_top_hgvsp

In [139]:
CBE_editing_master = pd.merge(CBE_editing_master, library[['gRNA_id', 'Gene']], on='gRNA_id')
ABE_editing_master = pd.merge(ABE_editing_master, library[['gRNA_id', 'Gene']], on='gRNA_id')

In [144]:
cols2 = ['gRNA_id', 'Gene', 'Total_sensor_reads', 'HGVSp', '%Reads', 'Codon',
    'Top_HGVSp', 'Top_HGVSp_Codon', 'Top_HGVSp_%Reads', 'WT_perc', 'Missense_perc',
    'Nonsense_perc',  'Mutation_Class_SAV',
    'Mutation_Class_Top_HGVSp', 'Editor',]

CBE_editing_master = CBE_editing_master[cols2]
ABE_editing_master = ABE_editing_master[cols2]
ABE_editing_master

,gRNA_id,Gene,Total_sensor_reads,HGVSp,%Reads,Codon,Top_HGVSp,Top_HGVSp_Codon,Top_HGVSp_%Reads,WT_perc,Missense_perc,Nonsense_perc,Mutation_Class_SAV,Mutation_Class_Top_HGVSp,Editor
0,gRNA_CDK19_targ_6273,CDK19,253727,Y502H,84.074616,502,Y502H,502,83.163400,14.614133,85.365767,0.020100,Missense,Missense,ABE
1,gRNA_CDK19_targ_6273,CDK19,253727,WT,14.614133,WT,Y502H,502,83.163400,14.614133,85.365767,0.020100,WT,Missense,ABE
2,gRNA_CDK19_targ_6274,CDK19,134773,Y502H,90.623493,502,Y502H,502,80.766177,3.828660,96.134241,0.037099,Missense,Missense,ABE
3,gRNA_CDK19_targ_6274,CDK19,134773,R501Q,9.617653,501,Y502H,502,80.766177,3.828660,96.134241,0.037099,Missense,Missense,ABE
4,gRNA_CDK19_targ_6274,CDK19,134773,WT,3.828660,WT,Y502H,502,80.766177,3.828660,96.134241,0.037099,WT,Missense,ABE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52644,gRNA_CDK6_targ_8477,CDK6,75049,M1V,85.890552,1,M1V,1,51.608949,12.792975,87.179043,0.027982,Missense,Missense,ABE
52645,gRNA_CDK6_targ_8477,CDK6,75049,K3E,17.800370,3,M1V,1,51.608949,12.792975,87.179043,0.027982,Missense,Missense,ABE
52646,gRNA_CDK6_targ_8477,CDK6,75049,E2G,16.153446,2,M1V,1,51.608949,12.792975,87.179043,0.027982,Missense,Missense,ABE
52647,gRNA_CDK6_targ_8477,CDK6,75049,WT,12.792975,WT,M1V,1,51.608949,12.792975,87.179043,0.027982,WT,Missense,ABE


In [ ]:
#CBE_editing_master.to_csv('../../screening_data/04_editing/CBE_editing_master.csv', index=False)
#ABE_editing_master.to_csv('../../screening_data/04_editing/ABE_editing_master.csv', index=False)

# compilation of LFC data

In [179]:
#and load in LFC/FDR information

#-----PLASMID BASE
ABE1_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/ABE_subpool1_barcode_Plasmid_base.csv')
CBE1_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/CBE_subpool1_barcode_Plasmid_base.csv')

#subpool1
ABE2_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/CDK12_13_ABE_barcode_Plasmid_base.csv')
ABE2_plas.columns = ABE2_plas.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE2_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/CDK12_13_CBE_barcode_Plasmid_base.csv')
CBE2_plas.columns = CBE2_plas.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#subpool3
ABE3_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/CDK2_4_6_ABE_barcode_Plasmid_base.csv')
ABE3_plas.columns = ABE3_plas.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE3_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/CDK2_4_6_CBE_barcode_Plasmid_base.csv')
CBE3_plas.columns = CBE3_plas.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#SY-5609
ABE_sy_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/SY_5609_ABE_barcode_Plasmid_base.csv')
ABE_sy_plas.columns = ABE_sy_plas.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE_sy_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/Plasmid_base/SY_5609_CBE_barcode_Plasmid_base.csv')
CBE_sy_plas.columns = CBE_sy_plas.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#iterative screen
#using proto counts for this
CBE_iterative_plas = pd.read_csv('../../screening_data/03_LFC_FDR_tables/proto_counts/Plasmid_base/Compound_mutant_proto_Plasmid_base.csv')


#----DMSO BASE
ABE1_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/ABE_subpool1_barcode_DMSO_base.csv')
CBE1_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/CBE_subpool1_barcode_DMSO_base.csv')

#subpool1
ABE2_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/CDK12_13_ABE_barcode_DMSO_base.csv')
ABE2_dmso.columns = ABE2_dmso.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE2_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/CDK12_13_CBE_barcode_DMSO_base.csv')
CBE2_dmso.columns = CBE2_dmso.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#subpool3
ABE3_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/CDK2_4_6_ABE_barcode_DMSO_base.csv')
ABE3_dmso.columns = ABE3_dmso.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE3_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/CDK2_4_6_CBE_barcode_DMSO_base.csv')
CBE3_dmso.columns = CBE3_dmso.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#SY-5609
ABE_sy_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/SY_5609_ABE_barcode_DMSO_base.csv')
ABE_sy_dmso.columns = ABE_sy_dmso.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE_sy_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/DMSO_base/SY_5609_CBE_barcode_DMSO_base.csv')
CBE_sy_dmso.columns = CBE_sy_dmso.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#iterative screen
#using proto counts for this
CBE_iterative_dmso = pd.read_csv('../../screening_data/03_LFC_FDR_tables/proto_counts/DMSO_base/Compound_mutant_proto_DMSO_base.csv')


#----T0 BASE
ABE1_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/ABE_subpool1_barcode_T0_base.csv')
CBE1_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/CBE_subpool1_barcode_T0_base.csv')

#subpool1
ABE2_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/CDK12_13_ABE_barcode_T0_base.csv')
ABE2_t0.columns = ABE2_t0.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE2_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/CDK12_13_CBE_barcode_T0_base.csv')
CBE2_t0.columns = CBE2_t0.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#subpool3
ABE3_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/CDK2_4_6_ABE_barcode_T0_base.csv')
ABE3_t0.columns = ABE3_t0.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE3_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/CDK2_4_6_CBE_barcode_T0_base.csv')
CBE3_t0.columns = CBE3_t0.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#SY-5609
ABE_sy_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/SY_5609_ABE_barcode_T0_base.csv')
ABE_sy_t0.columns = ABE_sy_t0.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

CBE_sy_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/barcode_counts/T0_base/SY_5609_CBE_barcode_T0_base.csv')
CBE_sy_t0.columns = CBE_sy_t0.columns.str.replace(r"(ABE_|CBE_)", "", regex=True)

#iterative screen
#using proto counts for this
CBE_iterative_t0 = pd.read_csv('../../screening_data/03_LFC_FDR_tables/proto_counts/T0_base/Compound_mutant_proto_T0_base.csv')


#------add identifiers to each screening dataset-----------
#----subpool 1-------
ABE1_plas['Screen_ID'] = 'Subpool1'
ABE1_dmso['Screen_ID'] = 'Subpool1'
ABE1_t0['Screen_ID'] = 'Subpool1'

ABE1_plas['Editor'] = 'ABE'
ABE1_dmso['Editor'] = 'ABE'
ABE1_t0['Editor'] = 'ABE'

CBE1_plas['Screen_ID'] = 'Subpool1'
CBE1_dmso['Screen_ID'] = 'Subpool1'
CBE1_t0['Screen_ID'] = 'Subpool1'

CBE1_plas['Editor'] = 'CBE'
CBE1_dmso['Editor'] = 'CBE'
CBE1_t0['Editor'] = 'CBE'
#----subpool 2-------
ABE2_plas['Screen_ID'] = 'Subpool2'
ABE2_dmso['Screen_ID'] = 'Subpool2'
ABE2_t0['Screen_ID'] = 'Subpool2'

ABE2_plas['Editor'] = 'ABE'
ABE2_dmso['Editor'] = 'ABE'
ABE2_t0['Editor'] = 'ABE'

CBE2_plas['Screen_ID'] = 'Subpool2'
CBE2_dmso['Screen_ID'] = 'Subpool2'
CBE2_t0['Screen_ID'] = 'Subpool2'

CBE2_plas['Editor'] = 'CBE'
CBE2_dmso['Editor'] = 'CBE'
CBE2_t0['Editor'] = 'CBE'

#----subpool 3-------
ABE3_plas['Screen_ID'] = 'Subpool3'
ABE3_dmso['Screen_ID'] = 'Subpool3'
ABE3_t0['Screen_ID'] = 'Subpool3'

ABE3_plas['Editor'] = 'ABE'
ABE3_dmso['Editor'] = 'ABE'
ABE3_t0['Editor'] = 'ABE'

CBE3_plas['Screen_ID'] = 'Subpool3'
CBE3_dmso['Screen_ID'] = 'Subpool3'
CBE3_t0['Screen_ID'] = 'Subpool3'

CBE3_plas['Editor'] = 'CBE'
CBE3_dmso['Editor'] = 'CBE'
CBE3_t0['Editor'] = 'CBE'

#----SY-5609-------
ABE_sy_plas['Screen_ID'] = 'SY-5609'
ABE_sy_dmso['Screen_ID'] = 'SY-5609'
ABE_sy_t0['Screen_ID'] = 'SY-5609'

ABE_sy_plas['Editor'] = 'ABE'
ABE_sy_dmso['Editor'] = 'ABE'
ABE_sy_t0['Editor'] = 'ABE'

CBE_sy_plas['Screen_ID'] = 'SY-5609'
CBE_sy_dmso['Screen_ID'] = 'SY-5609'
CBE_sy_t0['Screen_ID'] = 'SY-5609'

CBE_sy_plas['Editor'] = 'CBE'
CBE_sy_dmso['Editor'] = 'CBE'
CBE_sy_t0['Editor'] = 'CBE'

#----iterative------
CBE_iterative_plas['Screen_ID'] = 'Iterative'
CBE_iterative_dmso['Screen_ID'] = 'Iterative'
CBE_iterative_t0['Screen_ID'] = 'Iterative'

CBE_iterative_plas['Editor'] = 'CBE'
CBE_iterative_dmso['Editor'] = 'CBE'
CBE_iterative_t0['Editor'] = 'CBE'


In [180]:
#generating lists
Plasmid_ABE = [ABE1_plas, ABE2_plas, ABE3_plas, ABE_sy_plas]
Plasmid_CBE = [CBE1_plas, CBE2_plas, CBE3_plas, CBE_sy_plas, CBE_iterative_plas]
#should I exclude the SY DMSO condition? otherwise duplicate columns...

DMSO_ABE = [ABE1_dmso, ABE2_dmso, ABE3_dmso, ABE_sy_dmso]
DMSO_CBE = [CBE1_dmso, CBE2_dmso, CBE3_dmso, CBE_sy_dmso, CBE_iterative_dmso]

T0_ABE = [ABE1_t0, ABE2_t0, ABE3_t0, ABE_sy_t0]
T0_CBE = [CBE1_t0, CBE2_t0, CBE3_t0, CBE_sy_t0, CBE_iterative_t0]

In [190]:
def col_renamer(df_list, suffix):

    new_dfs = []
    for k in df_list:
        rep_cols = [i for i in k.columns if '_REP' in i]
        rep_cols_new = [f'{i}_{suffix}' for i in rep_cols]

        median_cols = [i for i in k.columns if '_median' in i]
        median_cols_new = [f'{i}_{suffix}' for i in median_cols]

        avg_cols = [i for i in k.columns if '_avg' in i]
        avg_cols_new = [f'{i}_{suffix}' for i in avg_cols]

        FDR_cols = [i for i in k.columns if 'FDR' in i]
        FDR_cols_new = [f'{i}_{suffix}' for i in FDR_cols]

        base_cols = ['base_RAW']
        base_cols_new = [f'base_RAW_{suffix}']

        old_cols = rep_cols + median_cols + avg_cols + FDR_cols + base_cols
        new_cols = rep_cols_new + median_cols_new + avg_cols_new + FDR_cols_new + base_cols_new

        d1 = dict(zip(old_cols, new_cols))

        new_df = k.rename(columns = d1)

        new_df = new_df[['gRNA_id', 'classification', 'Screen_ID', 'Editor'] + new_cols]

        new_df['Base_Value'] = suffix

        new_dfs.append(new_df)

    return new_dfs



In [191]:
Plasmid_ABE_new = col_renamer(Plasmid_ABE, 'Plasmid')
Plasmid_CBE_new = col_renamer(Plasmid_CBE, 'Plasmid')

DMSO_ABE_new = col_renamer(DMSO_ABE, 'DMSO')
DMSO_CBE_new = col_renamer(DMSO_CBE, 'DMSO')

T0_ABE_new = col_renamer(T0_ABE, 'T0')
T0_CBE_new = col_renamer(T0_CBE, 'T0')

In [196]:
ABE_combined = pd.concat(Plasmid_ABE_new + DMSO_ABE_new + T0_ABE_new)
CBE_combined = pd.concat(Plasmid_CBE_new + DMSO_CBE_new + T0_CBE_new)

In [ ]:
#ABE_combined.to_csv('../../screening_data/03_LFC_FDR_tables/ABE_master_LFC_table.csv', index=False)
#CBE_combined.to_csv('../../screening_data/03_LFC_FDR_tables/CBE_master_LFC_table.csv', index=False)